In [1]:
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Suppress warning messages during execution
warnings.filterwarnings('ignore')

print('====================================================================')
print('=== COMPONENT 1: SALES & FINANCIAL CREDIT READINESS ANALYTICS ===')
print('====================================================================\n')

=== COMPONENT 1: SALES & FINANCIAL CREDIT READINESS ANALYTICS ===



In [2]:
# =========================================================================
# STEP 1: DATA LOADING & PRELIMINARY CLEANING
# =========================================================================
try:
    df = pd.read_csv('sales & financial.csv')
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'monthly_revenue_rs': np.random.uniform(100000, 2000000, n),
        'monthly_expenses_rs': np.random.uniform(50000, 1500000, n),
        'digital_payment_ratio': np.random.uniform(0.1, 0.9, n),
        'months_active': np.random.randint(1, 60, n),
        'stockout_rate': np.random.uniform(0.0, 0.3, n),
        'profit_margin_pct': np.random.uniform(5, 35, n),
        'target_credit_ready': np.random.choice([0, 1], size=n, p=[0.4, 0.6]),
    })

df = df.drop_duplicates().reset_index(drop=True)
df = df.drop(columns=['shop_id', 'year_month'], errors='ignore')


In [3]:
# =========================================================================
# STEP 2: DOMAIN FEATURE ENGINEERING (PIPELINE COMPATIBLE)
# =========================================================================
def add_domain_features(X_df):
    """
    FunctionTransformer එක සඳහා Feature Engineering ශ්‍රිතය.
    Division by zero වැළැක්වීමට np.maximum() භාවිත කර ඇත.
    """
    X_out = X_df.copy()
    
    rev = np.maximum(X_out['monthly_revenue_rs'], 1.0) if 'monthly_revenue_rs' in X_out.columns else 1.0
    exp = X_out['monthly_expenses_rs'] if 'monthly_expenses_rs' in X_out.columns else 0.0
    active_m = np.maximum(X_out['months_active'], 1.0) if 'months_active' in X_out.columns else 1.0
    dig_ratio = X_out['digital_payment_ratio'] if 'digital_payment_ratio' in X_out.columns else 0.0
    
    X_out['net_cash_flow'] = rev - exp
    X_out['debt_to_income_ratio'] = exp / rev
    X_out['digital_revenue_volume'] = rev * dig_ratio
    X_out['revenue_per_active_month'] = rev / active_m
    X_out['cash_flow_margin'] = X_out['net_cash_flow'] / rev
    
    return X_out

# Feature Engineering Function Transformer
feature_engineer = FunctionTransformer(add_domain_features, validate=False)


In [4]:
# =========================================================================
# STEP 3: PREPROCESSING PIPELINE & DATA SPLIT
# =========================================================================
# Raw input data වෙන් කිරීම (Feature engineering සිදු කිරීමට පෙර)
X_raw = df.drop(
    columns=['target_credit_ready', 'recommended_loan_limit'], errors='ignore'
)
y_cls = df['target_credit_ready'].astype(int)

# Train/Test Split (Raw Features මත)
(
    X_train_raw,
    X_test_raw,
    y_train_cls,
    y_test_cls,
) = train_test_split(
    X_raw, y_cls, test_size=0.20, random_state=42, stratify=y_cls
)

# Target Regression calculation (Data Leakage වැළැක්වීමට Split කළ පසු)
X_train_engineered = add_domain_features(X_train_raw)
X_test_engineered = add_domain_features(X_test_raw)

y_train_reg = np.maximum(0, X_train_engineered['net_cash_flow'] * 3.5 * y_train_cls).astype(float)
y_test_reg = np.maximum(0, X_test_engineered['net_cash_flow'] * 3.5 * y_test_cls).astype(float)

# Preprocessing Columns හඳුනා ගැනීම
sample_feat = add_domain_features(X_raw)
num_cols = sample_feat.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = sample_feat.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imp', SimpleImputer(strategy='median')),
                ('sc', StandardScaler()),
            ]),
            num_cols,
        ),
        (
            'cat',
            Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]),
            cat_cols,
        ),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [5]:
# =========================================================================
# STEP 4: FULL PIPELINES (FEATURE ENG + PREPROC + CLASSIFIER)
# =========================================================================
models = {
    'LogisticRegression': Pipeline([
        ('feat_eng', feature_engineer),
        ('prep', preprocessor),
        (
            'clf',
            LogisticRegression(
                max_iter=1000, class_weight='balanced', random_state=42
            ),
        ),
    ]),
    'DecisionTree': Pipeline([
        ('feat_eng', feature_engineer),
        ('prep', preprocessor),
        (
            'clf',
            DecisionTreeClassifier(
                max_depth=6,
                min_samples_leaf=5,
                class_weight='balanced',
                random_state=42,
            ),
        ),
    ]),
    'RandomForest': Pipeline([
        ('feat_eng', feature_engineer),
        ('prep', preprocessor),
        (
            'clf',
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                class_weight='balanced',
                n_jobs=1,
                random_state=42,
            ),
        ),
    ]),
    'GradientBoosting': Pipeline([
        ('feat_eng', feature_engineer),
        ('prep', preprocessor),
        (
            'clf',
            GradientBoostingClassifier(
                n_estimators=250,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.85,
                random_state=42,
            ),
        ),
    ]),
}

try:
    from xgboost import XGBClassifier

    models['XGBoost'] = Pipeline([
        ('feat_eng', feature_engineer),
        ('prep', preprocessor),
        (
            'clf',
            XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.85,
                colsample_bytree=0.85,
                eval_metric='logloss',
                random_state=42,
            ),
        ),
    ])
except ImportError:
    print('Notice: XGBoost not installed. Running with Sklearn Models.')


In [6]:
# =========================================================================
# STEP 5: CROSS-VALIDATION BENCHMARKING & CHAMPION SELECTION
# =========================================================================
print('=== Running 5-Fold Cross Validation Benchmarking ===')
cv_results = {}

for name, m in models.items():
    scores = cross_validate(
        m,
        X_train_raw,
        y_train_cls,
        cv=cv,
        scoring=['roc_auc', 'accuracy', 'f1'],
        n_jobs=-1,
    )
    cv_results[name] = {
        'auc': scores['test_roc_auc'].mean(),
        'acc': scores['test_accuracy'].mean(),
        'f1': scores['test_f1'].mean(),
    }
    print(
        f'  {name:20s} | CV ROC-AUC: {cv_results[name]["auc"]:.3f} | CV Acc:'
        f' {cv_results[name]["acc"]:.3f} | CV F1: {cv_results[name]["f1"]:.3f}'
    )

best_model_name = max(cv_results, key=lambda k: cv_results[k]['auc'])
print(f'\n🏆 Champion Model Selected: {best_model_name}')

champion_cls_model = models[best_model_name]
champion_cls_model.fit(X_train_raw, y_train_cls)

y_pred = champion_cls_model.predict(X_test_raw)
y_proba = champion_cls_model.predict_proba(X_test_raw)[:, 1]

print('\n===================================================')
print(f'=== Champion Model ({best_model_name}) Test Performance ===')
print('===================================================')
print(f'Accuracy  : {accuracy_score(y_test_cls, y_pred):.3f}')
print(f'Precision : {precision_score(y_test_cls, y_pred):.3f}')
print(f'Recall    : {recall_score(y_test_cls, y_pred):.3f}')
print(f'F1-Score  : {f1_score(y_test_cls, y_pred):.3f}')
print(f'ROC-AUC   : {roc_auc_score(y_test_cls, y_proba):.3f}\n')
print(
    classification_report(
        y_test_cls, y_pred, target_names=['Not Credit Ready', 'Credit Ready']
    )
)


=== Running 5-Fold Cross Validation Benchmarking ===
  LogisticRegression   | CV ROC-AUC: 0.883 | CV Acc: 0.806 | CV F1: 0.808
  DecisionTree         | CV ROC-AUC: 0.790 | CV Acc: 0.724 | CV F1: 0.726
  RandomForest         | CV ROC-AUC: 0.857 | CV Acc: 0.777 | CV F1: 0.779
  GradientBoosting     | CV ROC-AUC: 0.859 | CV Acc: 0.779 | CV F1: 0.780
  XGBoost              | CV ROC-AUC: 0.858 | CV Acc: 0.777 | CV F1: 0.780

🏆 Champion Model Selected: LogisticRegression

=== Champion Model (LogisticRegression) Test Performance ===
Accuracy  : 0.833
Precision : 0.856
Recall    : 0.801
F1-Score  : 0.828
ROC-AUC   : 0.906

                  precision    recall  f1-score   support

Not Credit Ready       0.81      0.86      0.84       199
    Credit Ready       0.86      0.80      0.83       201

        accuracy                           0.83       400
       macro avg       0.83      0.83      0.83       400
    weighted avg       0.83      0.83      0.83       400



In [7]:
# =========================================================================
# STEP 6: REGRESSION MODEL TRAINING (CREDIT LIMIT PREDICTION)
# =========================================================================
print('\n=== Training Credit Limit Prediction Engine (Regression) ===')

credit_limit_regressor = Pipeline([
    ('feat_eng', feature_engineer),
    ('prep', preprocessor),
    (
        'reg',
        RandomForestRegressor(
            n_estimators=200,
            max_depth=8,
            random_state=42,
            n_jobs=1,
        ),
    ),
])

# Fit regressor only on Credit Ready training subset
credit_limit_regressor.fit(
    X_train_raw[y_train_cls == 1], y_train_reg[y_train_cls == 1]
)

# Evaluation on Credit Ready test set
X_test_raw_ready = X_test_raw[y_test_cls == 1]
y_test_reg_ready = y_test_reg[y_test_cls == 1]

y_reg_pred = credit_limit_regressor.predict(X_test_raw_ready)

print(
    'Regression Model MAE : LKR'
    f' {mean_absolute_error(y_test_reg_ready, y_reg_pred):,.2f}'
)
print(
    'Regression Model R2  :'
    f' {r2_score(y_test_reg_ready, y_reg_pred):.3f}'
)



=== Training Credit Limit Prediction Engine (Regression) ===
Regression Model MAE : LKR 143.18
Regression Model R2  : 1.000


In [8]:
# =========================================================================
# STEP 7: HYBRID ML + RULE ENGINE INTEGRATION (RAW DATA READY)
# =========================================================================
print('\n===================================================')
print('=== HYBRID ML & RULE ENGINE EVALUATION SIMULATION ===')
print('===================================================')

def evaluate_merchant_credit_decision(merchant_raw_row, cls_model, reg_model):
    """
    Raw merchant record එකක් ලබාගෙන Feature Engineering, ML Predictions 
    සහ Hard Business Rules එකවර පරීක්ෂා කර තීරණය ලබා දෙයි.
    """
    # Rule engine එක සඳහා අවශ්‍ය engineered metrics ගණනය කිරීම
    engineered_row = add_domain_features(merchant_raw_row)
    raw_data = engineered_row.iloc[0]

    prob_score = round(cls_model.predict_proba(merchant_raw_row)[0, 1] * 100)
    predicted_limit = max(0, round(reg_model.predict(merchant_raw_row)[0], -3))

    hard_blocks = []

    if raw_data.get('debt_to_income_ratio', 0) > 0.85:
        hard_blocks.append('CRITICAL: DTI Ratio exceeds safety threshold (>85%).')

    if raw_data.get('months_active', 100) < 3:
        hard_blocks.append('HIGH RISK: Business operating history is less than 3 months.')

    if raw_data.get('stockout_rate', 0) > 0.25:
        hard_blocks.append('OPERATIONAL RISK: Stockout rate exceeds 25%.')

    if hard_blocks:
        final_status = '🔴 REJECTED BY RULE ENGINE (Hard Block)'
        recommended_limit = 0
    elif prob_score >= 70:
        final_status = '🟢 APPROVED (Tier 1 Prime Merchant)'
        recommended_limit = predicted_limit
    elif prob_score >= 50:
        final_status = '🟡 CONDITIONAL APPROVAL (Tier 2 Micro-Loan)'
        recommended_limit = min(predicted_limit, 250000)
    else:
        final_status = '🔴 REJECTED (High Risk ML Score)'
        recommended_limit = 0

    return {
        'credit_score': f'{prob_score}/100',
        'final_status': final_status,
        'recommended_max_loan_lkr': f'LKR {recommended_limit:,.2f}',
        'rule_engine_alerts': hard_blocks if hard_blocks else ['None'],
    }

sample_merchant_raw = X_test_raw.iloc[[0]]
evaluation_result = evaluate_merchant_credit_decision(
    sample_merchant_raw, champion_cls_model, credit_limit_regressor
)

print(f'Merchant Reference Index : {X_test_raw.index[0]}')
print(f'Credit Readiness Score   : {evaluation_result["credit_score"]}')
print(f'Final Credit Status      : {evaluation_result["final_status"]}')
print(f'Max Safe Loan Amount     : {evaluation_result["recommended_max_loan_lkr"]}')
print(f'Rule Engine Alerts       : {evaluation_result["rule_engine_alerts"]}')



=== HYBRID ML & RULE ENGINE EVALUATION SIMULATION ===
Merchant Reference Index : 438
Credit Readiness Score   : 22/100
Final Credit Status      : 🔴 REJECTED (High Risk ML Score)
Max Safe Loan Amount     : LKR 0.00
Rule Engine Alerts       : ['None']


In [9]:
# =========================================================================
# STEP 8: MODEL SERIALIZATION
# =========================================================================
bundle = {
    'classifier_pipeline': champion_cls_model,
    'regressor_pipeline': credit_limit_regressor,
    'raw_feature_names': X_raw.columns.tolist(),
}

model_filename = 'component1_sales_financial_model.pkl'
joblib.dump(bundle, model_filename)
print(f'\n✅ Complete Upgraded Component 1 Bundle saved as "{model_filename}"')


✅ Complete Upgraded Component 1 Bundle saved as "component1_sales_financial_model.pkl"
